# Computational Analysis of Hypercube Graphs

## 1. Introduction

This notebook explores structural properties of hypercube graphs using
Python and NetworkX.

The n-dimensional hypercube graph Q_n has vertices represented by binary
strings of length n. Two vertices are adjacent if their binary
representations differ in exactly one coordinate.

The objectives of this notebook are to:

1. Construct hypercube graphs computationally.
2. Visualise small hypercubes.
3. Calculate their numbers of vertices and edges.
4. Examine vertex degrees and graph diameter.
5. Compare computational results with known theoretical expressions.

In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import math
import pandas as pd 

In [2]:

print("NetworkX version:", nx.__version__)

NetworkX version: 2.7.1


In [3]:
n=3
Q3 = nx.hypercube_graph(n)
print("Number of vertices:", Q3.number_of_nodes())
print("Number of edges:", Q3.number_of_edges())


Number of vertices: 8
Number of edges: 12


### Theoretical Verification

For an \(n\)-dimensional hypercube \(Q_n\), the number of vertices is

$$
|V(Q_n)| = 2^n
$$

and the number of edges is

$$
|E(Q_n)| = n2^{n-1}.
$$

For \(n=3\),

$$
|V(Q_3)| = 2^3 = 8
$$

and

$$
|E(Q_3)| = 3(2^2) = 12.
$$

Therefore, the computational results agree with the theoretical values.

In [4]:
def hypercube_properties(n):
    G = nx.hypercube_graph(n)

    properties = {
        "dimension": n,
        "vertices": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "diameter": nx.diameter(G),
        "regular_degree": next(iter(dict(G.degree()).values()))
    }

    return properties

In [5]:
hypercube_properties(3)

{'dimension': 3,
 'vertices': 8,
 'edges': 12,
 'diameter': 3,
 'regular_degree': 3}

In [6]:
for n in range(1, 7):
    print(hypercube_properties(n))

{'dimension': 1, 'vertices': 2, 'edges': 1, 'diameter': 1, 'regular_degree': 1}
{'dimension': 2, 'vertices': 4, 'edges': 4, 'diameter': 2, 'regular_degree': 2}
{'dimension': 3, 'vertices': 8, 'edges': 12, 'diameter': 3, 'regular_degree': 3}
{'dimension': 4, 'vertices': 16, 'edges': 32, 'diameter': 4, 'regular_degree': 4}
{'dimension': 5, 'vertices': 32, 'edges': 80, 'diameter': 5, 'regular_degree': 5}
{'dimension': 6, 'vertices': 64, 'edges': 192, 'diameter': 6, 'regular_degree': 6}


In [7]:
def theoretical_hypercube_properties(n):
    return{ "dimension": n,
          "vertices": 2**n,
          "edges": n*2**(n-1),
          "diameter": n,
          "regular_degree": n}

    

In [8]:
for n in range(1,7):
    print("properties:", theoretical_hypercube_properties(n) )

properties: {'dimension': 1, 'vertices': 2, 'edges': 1, 'diameter': 1, 'regular_degree': 1}
properties: {'dimension': 2, 'vertices': 4, 'edges': 4, 'diameter': 2, 'regular_degree': 2}
properties: {'dimension': 3, 'vertices': 8, 'edges': 12, 'diameter': 3, 'regular_degree': 3}
properties: {'dimension': 4, 'vertices': 16, 'edges': 32, 'diameter': 4, 'regular_degree': 4}
properties: {'dimension': 5, 'vertices': 32, 'edges': 80, 'diameter': 5, 'regular_degree': 5}
properties: {'dimension': 6, 'vertices': 64, 'edges': 192, 'diameter': 6, 'regular_degree': 6}


## 2. Distances in Hypercube Graphs

In a hypercube graph \(Q_n\), each vertex can be represented by a binary
string of length \(n\).

The graph distance between two vertices is equal to their Hamming distance,
which is the number of coordinates in which the two binary strings differ.

For example, in \(Q_3\),

$$
d(000,111)=3
$$

because the two vertices differ in all three coordinates, whereas

$$
d(000,001)=1.
$$

This section verifies these distance properties computationally.

In [9]:
Q3 = nx.hypercube_graph(3)

source = (0, 0, 0)

distances = nx.single_source_shortest_path_length(Q3, source)

for vertex, distance in sorted(distances.items()):
    print(f"{source} -> {vertex}: distance = {distance}")

(0, 0, 0) -> (0, 0, 0): distance = 0
(0, 0, 0) -> (0, 0, 1): distance = 1
(0, 0, 0) -> (0, 1, 0): distance = 1
(0, 0, 0) -> (0, 1, 1): distance = 2
(0, 0, 0) -> (1, 0, 0): distance = 1
(0, 0, 0) -> (1, 0, 1): distance = 2
(0, 0, 0) -> (1, 1, 0): distance = 2
(0, 0, 0) -> (1, 1, 1): distance = 3


In [10]:
from collections import Counter

distance_counts = Counter(distances.values())

print("Number of vertices at each distance from", source)

for distance in sorted(distance_counts):
    print(f"Distance {distance}: {distance_counts[distance]} vertices")

Number of vertices at each distance from (0, 0, 0)
Distance 0: 1 vertices
Distance 1: 3 vertices
Distance 2: 3 vertices
Distance 3: 1 vertices


### Metric-Degree Sequence

For a vertex \(v\), define

$$
d_i(v)=|\{u\in V(G):d(u,v)=i\}|,
$$

where \(d_i(v)\) is the number of vertices at distance \(i\) from \(v\).

For the hypercube \(Q_n\),

$$
d_i(v)=\binom{n}{i}.
$$

This is because a vertex at distance \(i\) from \(v\) must differ from
\(v\) in exactly \(i\) coordinates. There are

$$
\binom{n}{i}
$$

ways to choose those \(i\) coordinates.

Therefore, for \(Q_3\),

$$
( d_0(v), d_1(v), d_2(v), d_3(v) )
=
(1,3,3,1).
$$

In [11]:
def metric_degree_sequence_hypercube(n, vertex=None):
    """
    Compute the metric-degree sequence of a vertex in Q_n.
    """

    G = nx.hypercube_graph(n)

    # If no vertex is specified, select the first actual vertex
    # generated by NetworkX.
    if vertex is None:
        vertex = next(iter(G.nodes()))

    # Check that the selected vertex belongs to the graph
    if vertex not in G:
        raise ValueError(f"Vertex {vertex} is not a vertex of Q_{n}")

    distances = nx.single_source_shortest_path_length(G, vertex)
    counts = Counter(distances.values())

    sequence = [
        counts.get(i, 0)
        for i in range(nx.diameter(G) + 1)
    ]

    return sequence


In [12]:
for n in range(1, 7):
    sequence = metric_degree_sequence_hypercube(n)
    print(f"Q_{n}: {sequence}")

Q_1: [1, 1]
Q_2: [1, 2, 1]
Q_3: [1, 3, 3, 1]
Q_4: [1, 4, 6, 4, 1]
Q_5: [1, 5, 10, 10, 5, 1]
Q_6: [1, 6, 15, 20, 15, 6, 1]


In [13]:
def theoretical_metric_degree_sequence(n):
    return [math.comb(n, i) for i in range(n + 1)]

In [14]:
for n in range(1, 7):
    computational = metric_degree_sequence_hypercube(n)
    theoretical = theoretical_metric_degree_sequence(n)

    print(f"Q_{n}")
    print("Computational:", computational)
    print("Theoretical:  ", theoretical)
    print("Match:", computational == theoretical)
    print()

Q_1
Computational: [1, 1]
Theoretical:   [1, 1]
Match: True

Q_2
Computational: [1, 2, 1]
Theoretical:   [1, 2, 1]
Match: True

Q_3
Computational: [1, 3, 3, 1]
Theoretical:   [1, 3, 3, 1]
Match: True

Q_4
Computational: [1, 4, 6, 4, 1]
Theoretical:   [1, 4, 6, 4, 1]
Match: True

Q_5
Computational: [1, 5, 10, 10, 5, 1]
Theoretical:   [1, 5, 10, 10, 5, 1]
Match: True

Q_6
Computational: [1, 6, 15, 20, 15, 6, 1]
Theoretical:   [1, 6, 15, 20, 15, 6, 1]
Match: True



In [15]:
def all_vertices_same_metric_sequence(n):
    G = nx.hypercube_graph(n)

    sequences = []

    for vertex in G.nodes():
        sequence = metric_degree_sequence_hypercube(n, vertex)
        sequences.append(tuple(sequence))

    return len(set(sequences)) == 1

In [16]:
for n in range(1, 7):
    print(f"Q_{n}: {all_vertices_same_metric_sequence(n)}")

Q_1: True
Q_2: True
Q_3: True
Q_4: True
Q_5: True
Q_6: True


### Vertex Symmetry

The computational results show that every vertex of \(Q_n\) has the same
metric-degree sequence.

This is consistent with the vertex-transitive structure of the hypercube:
each vertex has the same local and distance-based structure.

Hence, the distance distribution around every vertex is identical.

In [17]:
import pandas as pd

results = []

for n in range(1, 7):
    G = nx.hypercube_graph(n)

    results.append({
        "Graph": f"Q_{n}",
        "Vertices": G.number_of_nodes(),
        "Edges": G.number_of_edges(),
        "Degree": n,
        "Diameter": nx.diameter(G),
        "Metric-Degree Sequence": str(
            metric_degree_sequence_hypercube(n)
        )
    })

results_df = pd.DataFrame(results)

results_df

,Graph,Vertices,Edges,Degree,Diameter,Metric-Degree Sequence
0,Q_1,2,1,1,1,"[1, 1]"
1,Q_2,4,4,2,2,"[1, 2, 1]"
2,Q_3,8,12,3,3,"[1, 3, 3, 1]"
3,Q_4,16,32,4,4,"[1, 4, 6, 4, 1]"
4,Q_5,32,80,5,5,"[1, 5, 10, 10, 5, 1]"
5,Q_6,64,192,6,6,"[1, 6, 15, 20, 15, 6, 1]"


## Conclusion

This computational investigation examined structural and distance-based
properties of the \(n\)-dimensional hypercube graph \(Q_n\).

The numerical results verified the theoretical relationships

$$
|V(Q_n)|=2^n,
$$

$$
|E(Q_n)|=n2^{n-1},
$$

$$
\deg(v)=n,
$$

and

$$
\mathrm{diam}(Q_n)=n.
$$

The distance analysis further showed that the number of vertices at
distance \(i\) from any vertex \(v\) is

$$
d_i(v)=\binom{n}{i}.
$$

Hence the metric-degree sequence of a vertex in \(Q_n\) corresponds to
the \(n\)-th row of Pascal's triangle.

The computational results obtained using Python and NetworkX agreed with
these theoretical expressions for the dimensions investigated.

This notebook serves as an introductory computational graph-theory study.
It is distinct from my MPhil research on graphs associated with algebraic
structures, which will be implemented separately in a later project.